# 07 — CamemBERTv2 Fine-Tuning (Phishing FR)

**Task 3.1** — Sicurre ML Pipeline  
**Compétences C6–C8** — Entraîner, évaluer et optimiser un modèle de ML

---

## Objectif

Fine-tuner **CamemBERTv2** (`almanach/camembertv2-base`, 110M params) pour la classification
binaire d'emails : `0` = légitime (ham), `1` = phishing.

**Pipeline :**
1. Charger le dataset final fusionné depuis `data/final/`
2. Tokenizer CamemBERTv2 (max 512 tokens)
3. Fine-tuning avec HuggingFace Trainer (3–5 epochs)
4. Évaluation : F1, précision, rappel, matrice de confusion
5. Export ONNX (INT8 quantized) pour production
6. Push vers HuggingFace Hub

**Runtime recommandé :** Google Colab (T4 GPU gratuit, ~20–40 min).  
**Deps locales :** gérées par `uv` via `pyproject.toml`.  
**Deps Colab :** installées via `pip` dans la première cellule.

In [ ]:
# ── Environment Detection & Install ──────────────────────────────────
import sys

IN_COLAB: bool = "google.colab" in sys.modules

if IN_COLAB:
    print("🔧 Running on Google Colab — installing dependencies...")
    !pip install -q transformers datasets evaluate accelerate scikit-learn \
        seaborn matplotlib onnx onnxruntime optimum sentencepiece
    print("✅ Dependencies installed")
else:
    print("💻 Running locally — using uv-managed deps from pyproject.toml")
    print("   Run `uv sync` if you haven't already.")

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

import torch

# ── Device Detection ──────────────────────────────────────────────────
DEVICE: str = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"🖥️  Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────
MODEL_NAME: str = "almanach/camembertv2-base"  # 110M params, French RoBERTa
MAX_LENGTH: int = 512
NUM_LABELS: int = 2
LABEL_NAMES: list[str] = ["legitimate", "phishing"]

# Training hyperparameters
BATCH_SIZE: int = 16 if DEVICE == "cuda" else 8
NUM_EPOCHS: int = 4
LEARNING_RATE: float = 2e-5
WEIGHT_DECAY: float = 0.01
WARMUP_RATIO: float = 0.1

# Paths
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/sicurre/data/final")
    OUTPUT_DIR = Path("/content/drive/MyDrive/sicurre/data/models/camembertv2-phishing-fr")
else:
    DATA_DIR = Path("../data/final")
    OUTPUT_DIR = Path("../data/models/camembertv2-phishing-fr")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model       : {MODEL_NAME}")
print(f"Max length  : {MAX_LENGTH}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Epochs      : {NUM_EPOCHS}")
print(f"LR          : {LEARNING_RATE}")
print(f"Data dir    : {DATA_DIR}")
print(f"Output dir  : {OUTPUT_DIR}")

## 1. Load Dataset

Expected CSV format in `data/final/{train,val,test}/`:

| Column | Type | Description |
|--------|------|-------------|
| `text` | str | Email body (anonymized, French) |
| `label` | int | 0 = legitimate, 1 = phishing |
| `source` | str | Origin dataset identifier |
| `language` | str | `fr` or `en` |

In [ ]:
# ── Load Data Splits ──────────────────────────────────────────────────
def load_split(split_dir: Path) -> pd.DataFrame:
    """Load all CSVs from a split directory and concatenate."""
    csvs = list(split_dir.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in {split_dir}")
    dfs = [pd.read_csv(f, encoding="utf-8") for f in csvs]
    df = pd.concat(dfs, ignore_index=True)
    # Validate required columns
    assert "text" in df.columns, "Missing 'text' column"
    assert "label" in df.columns, "Missing 'label' column"
    return df


df_train = load_split(DATA_DIR / "train")
df_val = load_split(DATA_DIR / "val")
df_test = load_split(DATA_DIR / "test")

print(f"── Dataset Sizes ──")
print(f"   Train : {len(df_train):,} rows")
print(f"   Val   : {len(df_val):,} rows")
print(f"   Test  : {len(df_test):,} rows")
print(f"   Total : {len(df_train) + len(df_val) + len(df_test):,} rows")

print(f"\n── Label Distribution (Train) ──")
print(df_train["label"].value_counts().rename({0: "legitimate", 1: "phishing"}))

In [ ]:
# ── Convert to HuggingFace Datasets ──────────────────────────────────
ds = DatasetDict({
    "train": Dataset.from_pandas(df_train[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(df_val[["text", "label"]], preserve_index=False),
    "test": Dataset.from_pandas(df_test[["text", "label"]], preserve_index=False),
})

print(ds)
print(f"\n── Sample ──")
print(ds["train"][0])

## 2. Tokenization

CamemBERTv2 uses a SentencePiece tokenizer. We truncate/pad to `MAX_LENGTH=512` tokens.

In [ ]:
# ── Load Tokenizer ────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer   : {type(tokenizer).__name__}")
print(f"Vocab size  : {tokenizer.vocab_size:,}")
print(f"Max length  : {MAX_LENGTH}")


def tokenize_fn(examples: dict) -> dict:
    """Tokenize text with truncation and padding."""
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )


# Apply tokenization (batched for speed)
ds_tokenized = ds.map(tokenize_fn, batched=True, batch_size=1000)

# Set PyTorch format
ds_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"\n✅ Tokenized — columns: {ds_tokenized['train'].column_names}")
print(f"   Sample input_ids shape: {ds_tokenized['train'][0]['input_ids'].shape}")

## 3. Model Setup

Load `almanach/camembertv2-base` with a classification head (2 labels).
The pre-trained LM head is replaced by a randomly initialized linear layer.

In [ ]:
# ── Load Pre-trained Model ────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={0: "legitimate", 1: "phishing"},
    label2id={"legitimate": 0, "phishing": 1},
)

# Count parameters
total_params: int = sum(p.numel() for p in model.parameters())
trainable_params: int = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,}")
print(f"   Labels           : {model.config.id2label}")

## 4. Training

HuggingFace `Trainer` with:
- AdamW optimizer, linear warmup (10%)
- Evaluation every epoch
- Early stopping (patience=2) on validation F1
- Best model checkpoint saved automatically

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred) -> dict:
    """Compute F1, precision, recall, and accuracy for binary classification."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "f1": f1_metric.compute(predictions=predictions, references=labels)["f1"],
        "precision": precision_metric.compute(predictions=predictions, references=labels)["precision"],
        "recall": recall_metric.compute(predictions=predictions, references=labels)["recall"],
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
    }

In [ ]:
# ── Training Arguments ────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    
    # Training
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    
    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Logging
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    report_to="none",  # Disable W&B / MLflow for now
    
    # Performance
    fp16=DEVICE == "cuda",  # Mixed precision on GPU
    dataloader_num_workers=2 if DEVICE != "cpu" else 0,
    
    # Reproducibility
    seed=42,
)

print(f"✅ Training config ready")
print(f"   Epochs       : {NUM_EPOCHS}")
print(f"   Batch size   : {BATCH_SIZE}")
print(f"   FP16         : {training_args.fp16}")
print(f"   Eval strategy: {training_args.eval_strategy}")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tokenized["train"],
    eval_dataset=ds_tokenized["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Starting training...")
train_result = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Total steps  : {train_result.global_step}")
print(f"   Training loss: {train_result.training_loss:.4f}")

## 5. Evaluation on Test Set

Held-out test set evaluation (never seen during training or validation).

In [ ]:
# ── Evaluate on Test Set ──────────────────────────────────────────────
test_results = trainer.evaluate(ds_tokenized["test"])

print("── Test Set Results ──")
for key, value in sorted(test_results.items()):
    if key.startswith("eval_"):
        metric_name = key.replace("eval_", "")
        if isinstance(value, float):
            print(f"   {metric_name:12s}: {value:.4f}")

# Target: F1 ≥ 0.95
f1_score: float = test_results.get("eval_f1", 0.0)
if f1_score >= 0.95:
    print(f"\n🎯 F1 = {f1_score:.4f} — TARGET MET (≥ 0.95)")
else:
    print(f"\n⚠️  F1 = {f1_score:.4f} — below target (0.95). Consider more data or hyperparameter tuning.")

In [ ]:
# ── Detailed Classification Report ───────────────────────────────────
predictions = trainer.predict(ds_tokenized["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print("── Classification Report ──")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=LABEL_NAMES,
    yticklabels=LABEL_NAMES,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("CamemBERTv2 — Confusion Matrix (Test Set)")
plt.tight_layout()

# Save figure
fig_path = OUTPUT_DIR / "confusion_matrix.png"
fig.savefig(fig_path, dpi=150)
print(f"✅ Saved: {fig_path}")
plt.show()

## 6. Training History Visualization

In [ ]:
# ── Plot Training History ─────────────────────────────────────────────
log_history = trainer.state.log_history

# Extract epoch-level metrics
train_losses = [(h["epoch"], h["loss"]) for h in log_history if "loss" in h and "eval_loss" not in h]
eval_entries = [h for h in log_history if "eval_f1" in h]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
if train_losses:
    epochs_t, losses_t = zip(*train_losses)
    axes[0].plot(epochs_t, losses_t, "o-", label="Train", alpha=0.7)
if eval_entries:
    axes[0].plot(
        [h["epoch"] for h in eval_entries],
        [h["eval_loss"] for h in eval_entries],
        "s-", label="Validation", alpha=0.7,
    )
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 curve
if eval_entries:
    axes[1].plot(
        [h["epoch"] for h in eval_entries],
        [h["eval_f1"] for h in eval_entries],
        "s-", color="green", label="F1",
    )
    axes[1].axhline(y=0.95, color="red", linestyle="--", label="Target (0.95)", alpha=0.5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1 Score")
axes[1].set_title("Validation F1")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "training_history.png", dpi=150)
print(f"✅ Saved: {OUTPUT_DIR / 'training_history.png'}")
plt.show()

## 7. Save Model & Tokenizer

Save the best model locally, then export to ONNX for production inference.

In [ ]:
# ── Save Best Model ───────────────────────────────────────────────────
model_save_path: Path = OUTPUT_DIR / "best_model"
trainer.save_model(str(model_save_path))
tokenizer.save_pretrained(str(model_save_path))

print(f"✅ Model saved: {model_save_path}")
print(f"   Contents: {[f.name for f in model_save_path.iterdir()]}")

## 8. ONNX Export (Production)

Export to ONNX format for fast CPU inference with ONNX Runtime (see ADR-0002).
This is the format used by `phishing-api` in production.

In [ ]:
# ── ONNX Export ───────────────────────────────────────────────────────
try:
    from optimum.onnxruntime import ORTModelForSequenceClassification
    from optimum.onnxruntime.configuration import AutoQuantizationConfig
    from optimum.onnxruntime import ORTQuantizer

    onnx_path: Path = OUTPUT_DIR / "onnx"
    
    # Export to ONNX
    ort_model = ORTModelForSequenceClassification.from_pretrained(
        str(model_save_path),
        export=True,
    )
    ort_model.save_pretrained(str(onnx_path))
    tokenizer.save_pretrained(str(onnx_path))

    print(f"✅ ONNX model exported: {onnx_path}")
    
    # INT8 Quantization
    quantizer = ORTQuantizer.from_pretrained(str(onnx_path))
    qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False)
    quantizer.quantize(save_dir=str(onnx_path / "quantized"), quantization_config=qconfig)
    
    print(f"✅ INT8 quantized model saved: {onnx_path / 'quantized'}")
    
    # Size comparison
    onnx_size = sum(f.stat().st_size for f in onnx_path.rglob("*.onnx")) / 1024 / 1024
    quant_size = sum(f.stat().st_size for f in (onnx_path / "quantized").rglob("*.onnx")) / 1024 / 1024
    print(f"   ONNX size     : {onnx_size:.1f} MB")
    print(f"   Quantized size: {quant_size:.1f} MB")
    print(f"   Compression   : {(1 - quant_size/onnx_size)*100:.0f}%")

except ImportError:
    print("⚠️  `optimum` not installed. Skipping ONNX export.")
    print("   Install with: pip install optimum[onnxruntime]")

## 9. Push to HuggingFace Hub (Optional)

Push the fine-tuned model for team access and deployment.

In [ ]:
# ── Push to HuggingFace Hub ───────────────────────────────────────────
PUSH_TO_HUB: bool = False  # Set to True when ready
HUB_REPO_ID: str = "sicurre/camembertv2-phishing-fr"  # Change to your org/repo

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()  # Will prompt for token
    
    trainer.push_to_hub(
        repo_id=HUB_REPO_ID,
        commit_message=f"Fine-tuned CamemBERTv2 for French phishing detection (F1={f1_score:.4f})",
    )
    print(f"✅ Pushed to https://huggingface.co/{HUB_REPO_ID}")
else:
    print("ℹ️  Hub push disabled. Set PUSH_TO_HUB = True when ready.")

## 10. Quick Inference Test

Sanity check: classify a few example emails with the fine-tuned model.

In [ ]:
# ── Quick Inference ───────────────────────────────────────────────────
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if DEVICE == "cuda" else -1,
)

# Test examples (French)
test_emails: list[str] = [
    # Phishing
    "Bonjour, votre compte bancaire a été bloqué. Cliquez ici pour vérifier votre identité immédiatement : http://banque-secure.xyz/login",
    # Phishing
    "URGENT : Votre colis n°FR-2847 est en attente. Payez 2,99€ de frais de douane pour le recevoir. Lien sécurisé : bit.ly/colis-fr",
    # Legitimate
    "Bonjour Madame Dupont, je vous confirme notre rendez-vous de mardi à 14h dans nos locaux. Cordialement, Jean Martin",
    # Legitimate
    "Votre facture EDF n°2024-0847 d'un montant de 127,34€ est disponible dans votre espace client. Aucune action requise.",
]

print("── Inference Results ──")
for email in test_emails:
    result = classifier(email, truncation=True, max_length=MAX_LENGTH)[0]
    label = result["label"]
    score = result["score"]
    emoji = "🚨" if label == "phishing" else "✅"
    print(f"\n{emoji} {label} (confidence: {score:.4f})")
    print(f"   {email[:80]}..." if len(email) > 80 else f"   {email}")

## Résumé

| Métrique | Valeur |
|----------|--------|
| Modèle de base | `almanach/camembertv2-base` (110M params) |
| Task | Binary classification (legitimate / phishing) |
| Dataset | French phishing corpus from `data/final/` |
| Best F1 (test) | See cell 13 above |
| Export | ONNX + INT8 quantized (production-ready) |
| Artifacts | `data/models/camembertv2-phishing-fr/` |

**Compétences démontrées :**
- **C6** : Entraîner un modèle de classification (fine-tuning CamemBERTv2)
- **C7** : Évaluer les performances (F1, precision, recall, confusion matrix)
- **C8** : Optimiser et exporter (ONNX INT8 quantization)
- **C9** : Reproductibilité (seed, saved checkpoints, logged metrics)

**Next steps :**
- Hyperparameter search if F1 < 0.95
- Error analysis on misclassified examples
- Integration test with `phishing-api` ONNX Runtime endpoint